In [68]:
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt
import os
!pip install rapidfuzz
from rapidfuzz import process
import pickle

  Obtaining dependency information for rapidfuzz from https://files.pythonhosted.org/packages/cf/99/5fa23e204435803875daefda73fd61baeabc3c36b8fc0e34c1705aab8c7b/rapidfuzz-3.14.3-cp311-cp311-win_amd64.whl.metadata
   ---------------------------------------- 0.0/1.5 MB ? eta -:--:--
   - -------------------------------------- 0.1/1.5 MB 2.0 MB/s eta 0:00:01
   -------- ------------------------------- 0.3/1.5 MB 4.2 MB/s eta 0:00:01
   ---------- ----------------------------- 0.4/1.5 MB 4.4 MB/s eta 0:00:01
   ------------------------- -------------- 1.0/1.5 MB 5.6 MB/s eta 0:00:01
   ------------------------------- -------- 1.2/1.5 MB 5.5 MB/s eta 0:00:01
   ------------------------------------- -- 1.5/1.5 MB 5.8 MB/s eta 0:00:01
   ------------------------------------- -- 1.5/1.5 MB 5.8 MB/s eta 0:00:01
   ---------------------------------------- 1.5/1.5 MB 4.5 MB/s eta 0:00:00


In [ ]:
# Count number of species per aviary 
metadata_path = 'metadata_aviaries/'
species_df = pd.read_csv(metadata_path + 'Avieries_obsolete.csv', encoding='latin-1')

number_species = species_df.groupby('Aviary')['Common name'].nunique()
number_individuals = species_df.groupby('Aviary')['Total count'].sum()
metadata_paths = species_df.groupby('Aviary')['preprocessed data'].first()  

combined_df = pd.DataFrame({'Number of Species': number_species, 'Number of Individuals': number_individuals, "preprocessed data": metadata_paths})
combined_df = combined_df.dropna() # Some aviaries don't have metadata files, so we drop those rows for now

# Link preprocessed data path to metadata path as they are not in the same format
aviaries_metadata = [f for f in os.listdir(metadata_path) if f.endswith('.xlsx')]
for idx, row in combined_df.iterrows():
    avirary_path = row["preprocessed data"]
    best_match = process.extractOne(avirary_path, aviaries_metadata)
    if best_match and best_match[1] > 80:
        file_path = best_match[0]

    combined_df.loc[row.name, "Metadata_filename"] = file_path

combined_df.reset_index(inplace=True)
combined_df


['fl_avifauna_flamingos_sept25_data_meta.xlsx', 'fl_avifauna_storage_ibises_sept25_data_meta.xlsx', 'fl_avifauna_vultures_sept25_data_meta.xlsx', 'fl_beekse_bergen_20250404_meta.xlsx', 'fl_beekse_bergen_20250412_meta.xlsx', 'fl_blijdorp_flamingos_dec2025_data_meta.xlsx', 'fl_cologne_zoo_flamingos_nov2025_data_meta.xlsx', 'fl_gaia_zoo_congo_15aug25_data_meta.xlsx', 'fl_gaia_zoo_savannah_08aug25_data_meta.xlsx', 'fl_gaia_zoo_taiga_18Jul25_data_meta.xlsx', 'fl_zoo_eindhoven_20250308_meta.xlsx', 'fl_zoo_eindhoven_20250315_meta.xlsx', 'fl_zoo_eindhoven_20250426_meta.xlsx', 'fl_zoo_eindhoven_20250503_meta.xlsx', 'fl_zoo_helsinki_20250624_meta.xlsx', 'fl_zoo_helsinki_20250701_meta.xlsx', 'fl_zoo_parc_aug25_data_meta.xlsx']


,Aviary,Number of Species,Number of Individuals,preprocessed data,Metadata_filename
0,"Avifauna, Cuba aviary",8,247,fl_avifauna_flamingos_sept2025_data,fl_avifauna_flamingos_sept25_data_meta.xlsx
1,"Avifauna, Umvikeli aviary",13,71,fl_avifauna_vultures_sept2025_data,fl_avifauna_vultures_sept25_data_meta.xlsx
2,"Avifauna, storage",4,36,fl_avifauna_storage_sept2025_data,fl_avifauna_vultures_sept25_data_meta.xlsx
3,"Beekse Bergen, Aviary 2",1,2,fl_beekse_bergen_20250404,fl_beekse_bergen_20250404_meta.xlsx
4,"Beekse Bergen, Aviary 3",8,163,fl_beekse_bergen_20250404,fl_beekse_bergen_20250404_meta.xlsx
5,"Beekse Bergen, Aviary 4 (Africa)",8,182,fl_beekse_bergen_20250412,fl_beekse_bergen_20250412_meta.xlsx
6,"Beekse Bergen, aviary 5",2,113,fl_beekse_bergen_flaminogos_sept2025_data,fl_beekse_bergen_20250412_meta.xlsx
7,"Blijdorp, Flamingos",4,204,fl_blijdorp_flamingos_dec2025_data,fl_blijdorp_flamingos_dec2025_data_meta.xlsx
8,"Cologne, Flamingos",1,153,fl_cologne_zoo_flamingos_nov2025_data,fl_cologne_zoo_flamingos_nov2025_data_meta.xlsx
9,"Cologne, Hippodom",17,98,fl_cologne_zoo_indoors_long_nov2025_data,fl_cologne_zoo_flamingos_nov2025_data_meta.xlsx


In [72]:
# count number of bird call per aviary meta data file
# We don't have all the metadata files, so we can't get the info for all aviaries in the cell above

aviaries_metadata = [f for f in os.listdir(metadata_path) if f.endswith('.xlsx')]
EDA_df = pd.DataFrame(columns=["file_name", "call_count", "total_entries", "Number of Species", "Number of Individuals"])

for file in aviaries_metadata:
    df = pd.read_excel(metadata_path + file)

    print(combined_df.loc[combined_df["Metadata_filename"] == file])

    call_count = df["fusion_model_prediction"].notnull().sum()
    
    number_species = combined_df.loc[combined_df["Metadata_filename"] == file, "Number of Species"].iloc[0] if file in combined_df["Metadata_filename"].values else 0
    number_individuals = combined_df.loc[combined_df["Metadata_filename"] == file, "Number of Individuals"].iloc[0] if file in combined_df["Metadata_filename"].values else 0
    
    new_row = {
        "file_name": file,
        "call_count": call_count,
        "total_entries": len(df),
        "Number_Species": number_species,
        "Number_Individuals": number_individuals
    }
    EDA_df.loc[len(EDA_df)] = new_row
    
print(EDA_df.head())

                  Aviary  Number of Species  Number of Individuals  \
0  Avifauna, Cuba aviary                  8                    247   

                     preprocessed data  \
0  fl_avifauna_flamingos_sept2025_data   

                             Metadata_filename  
0  fl_avifauna_flamingos_sept25_data_meta.xlsx  
Empty DataFrame
Columns: [Aviary, Number of Species, Number of Individuals, preprocessed data, Metadata_filename]
Index: []
                      Aviary  Number of Species  Number of Individuals  \
1  Avifauna, Umvikeli aviary                 13                     71   
2          Avifauna, storage                  4                     36   

                    preprocessed data  \
1  fl_avifauna_vultures_sept2025_data   
2   fl_avifauna_storage_sept2025_data   

                            Metadata_filename  
1  fl_avifauna_vultures_sept25_data_meta.xlsx  
2  fl_avifauna_vultures_sept25_data_meta.xlsx  
                    Aviary  Number of Species  Number of Indi

In [73]:
EDA_df.to_csv("Aviary_EDA.csv", index=False)